# Company research: permitted-source data workflow

Use the instructor workbook, Capital IQ workbook and `Research_Data.csv.gz` from the latest Release. All restricted academic-database data and their old dependent rankings are excluded; newly recomputed permitted-source provisional rankings are included. No credentials are stored here. Other providers require their own authorized access; local access does not imply public redistribution permission.


In [ ]:
from pathlib import Path
import csv,json,gzip,os
import pandas as pd
csv.field_size_limit(2**31-1)
DATA=Path("Research_Data.csv.gz")
assert DATA.exists(), "Download the current Release CSV beside this notebook."
def records(dataset=None):
    with gzip.open(DATA,"rt",encoding="utf-8-sig",newline="") as f:
        for row in csv.DictReader(f):
            if dataset is None or row["dataset"]==dataset:
                yield row
def table(dataset, limit=200):
    from itertools import islice
    return pd.DataFrame(json.loads(r["record_json"]) for r in islice(records(dataset), limit))
from collections import Counter
print(Counter(r["dataset"] for r in records()))


## View research, sources and permitted target inputs
Profile rows preserve exact evidence text and source links. A missing value remains unknown. General and industry updates are separate record types linked by company ID. Scores withdrawn after the source change are not silently reused.

Notebook tables default to a 200-record sample to keep memory use manageable; use the streaming records() helper for the full corpus.


In [ ]:
profiles=table("Unified_Public_Research")
display(profiles[[c for c in ["key","name","main_filled","industry_filled"] if c in profiles]].head())
inputs=table("Restricted_Target_Inputs", limit=560)
display(inputs.head())
if len(profiles):
    profile=profiles.iloc[0].to_dict()
    display(pd.DataFrame(profile.get("sources",{})).T)
    display(pd.DataFrame(profile.get("evidence",{})).T)


## Original workbooks
Only the two permitted supplied workbooks are inputs. Retain file, sheet and cell provenance for imported observations.


In [ ]:
for name in ["Company_List_for_Interns.xlsx","CapitalIQ_560.xlsx"]:
    p=Path(name)
    if p.exists():
        print(name,pd.ExcelFile(p).sheet_names)


## Source retrieval examples
SEC supplies reported facts, not consensus estimates. LSEG refresh requires an authorized local session and separately entitled fields. Target SEC data is research-only; it is never an input to simulated-private matching or forecasts. Functions below do not run automatically.


In [ ]:
import requests
def fetch_sec_facts(cik):
    contact=os.environ.get("SEC_USER_AGENT")
    if not contact: raise RuntimeError("Configure your contact SEC_USER_AGENT locally.")
    url=f"https://data.sec.gov/api/xbrl/companyfacts/CIK{int(cik):010d}.json"
    response=requests.get(url,headers={"User-Agent":contact},timeout=60)
    response.raise_for_status()
    return {"source_url":url,"record":response.json()}
def fetch_lseg(instruments,fields):
    import lseg.data as ld
    ld.open_session()
    try: return ld.get_data(universe=instruments,fields=fields)
    finally: ld.close_session()


## Rebuild and validate publication data
The active comparable rebuild is `code/rebuild_permitted_comparables.py`; package it with `code/package_rebuilt_comparables.py`. It uses only permitted evidence and frozen 60/40 general/industry weights. Validate with `code/validate_rebuilt_comparables.py` and `code/exclude_restricted_sources.py --validate`; do not rerun the retired cleanup generator on rebuilt results. `code/build_unified_research_site.py` checks the source-filtered viewer. Five-year forecasts remain a plan in `code/04_Revenue_Growth_Methodology.md`.


## Inspect newly rebuilt comparisons
`Public_Comparables` holds candidate order, descriptions and provisional indices. `Comparable_Facet_Scores` holds weights, indices, contributions and evidence IDs. Join those IDs to `Matching_Facet_Evidence` for the exact passages and source links. Unknowns remain missing; these indices are not calibrated probabilities. Stream one target at a time to avoid loading the full text corpus into memory.

In [ ]:
target_id = "IQ23557"
comparisons = pd.DataFrame([json.loads(r["record_json"]) for r in records("Public_Comparables") if r["company_id"] == target_id])
display(comparisons[[c for c in ["Candidate Order","Candidate Name","Ranking Score","Similarity Percent","Evidence Coverage Percent"] if c in comparisons]].head(10))


## Executed private growth prototype
Netchex revenue is indexed to a hypothetical 2026 reference of 100. Peer guidance, fiscal/calendar assumptions and stress settings are explicit. Actual private financials stay unknown. EBITDA/EV records are conditional sensitivities, not monetary forecasts; the other seven GrowthCurve companies remain blocked. No calibrated accuracy or private discount is claimed.

In [ ]:
forecast_data = {}
for row in records():
    if row["dataset"].startswith("Forecast_"):
        forecast_data.setdefault(row["dataset"], []).append(json.loads(row["record_json"]))
forecast = pd.DataFrame(forecast_data.get("Forecast_Scenario", []))
if len(forecast):
    display(forecast.pivot(index="calendar_year", columns="scenario", values="revenue_index").round(2))
    display(pd.DataFrame(forecast_data.get("Forecast_Assumption", [])))
    display(pd.DataFrame(forecast_data.get("Forecast_Readiness", []))[["company", "status", "reason"]])
else:
    print("Download the forecast-prototype Release CSV to view these records.")
